# text (CLIP) conditioned Pokémon DDPM —— annotated version (exp06)

This version replaces "discrete name lookup" with a **frozen CLIP text encoder + a small projection layer**, using semantic attribute descriptions
(`"Pikachu, a yellow electric-type mouse pokémon, ..."`) as the condition. Below we **break it down block by block**.

Overall it's still DDPM:
- **Forward**: add noise to a clean image step by step into pure noise (fixed math)
- **Reverse**: train a **UNet** to denoise; at each step it predicts "the noise ε that was added"
- **Condition**: tell the UNet "how much noise now (time t)" + "what to draw (text semantics)"

**Two stages**: Stage 1 learns general logic on all generations → Stage 2 fine-tunes on the 20.
(Does not touch your existing `pokemon_train.ipynb` / `pokemon_gen.ipynb`.)

## 1. Imports + timestep embedding

The denoising network needs to know at each step "which step it is now, how much noise". Encode the integer timestep `t` with **sine/cosine encoding**
into a vector (same idea as Transformer positional encoding: sin/cos at different frequencies let the network distinguish different t).

In [ ]:
import math, os, glob, json
import torch
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"

def sinusoidal_embedding(t, dim):
    """t: (B,) integer timesteps -> (B, dim) float embedding"""
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / half)
    args = t[:, None].float() * freqs[None, :]
    return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

## 2. ResBlock

The smallest building block of the UNet. Each ResBlock:
- Two blocks of `GroupNorm → SiLU → Conv3×3` (main body, extracts local features)
- In the middle, add the **condition vector `c`** (time + text semantics) to the feature map —— this is the "condition injection" point
- Outermost **residual connection** `+ x` (makes deep networks easier to train: learn the "delta" rather than the whole mapping)
- When the channel count changes, the residual branch uses a `1×1` convolution to align (`self.skip`)

Parameter `c_dim` = condition vector dimension (we use 256).

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, c_dim, groups=8):
        super().__init__()
        self.block1 = nn.Sequential(nn.GroupNorm(groups, in_ch), nn.SiLU(),
                                    nn.Conv2d(in_ch, out_ch, 3, padding=1))
        self.cond_mlp = nn.Sequential(nn.SiLU(), nn.Linear(c_dim, out_ch))   # condition -> channel bias
        self.block2 = nn.Sequential(nn.GroupNorm(groups, out_ch), nn.SiLU(),
                                    nn.Conv2d(out_ch, out_ch, 3, padding=1))
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x, c):
        h = self.block1(x)
        h = h + self.cond_mlp(c)[:, :, None, None]   # inject condition c (broadcast to every pixel)
        h = self.block2(h)
        return h + self.skip(x)                       # residual connection

## 3. Self-Attention AttnBlock

Convolution only sees local neighborhoods; Self-Attention lets **each pixel see the whole image**, improving overall structure/symmetry.
The cost is O(N²) (N=number of pixels), so we **only place it at low-resolution layers** (24², 12²); high resolution is too expensive to use.

In [ ]:
class AttnBlock(nn.Module):
    def __init__(self, ch, heads=4):
        super().__init__()
        self.norm = nn.GroupNorm(8, ch)
        self.mha = nn.MultiheadAttention(ch, heads, batch_first=True)
    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x).reshape(B, C, H*W).transpose(1, 2)   # (B, num_pixels, channels) treated as a sequence
        h, _ = self.mha(h, h, h)                              # self-attention
        return x + h.transpose(1, 2).reshape(B, C, H, W)      # residual

## 4. Stage (one resolution level)

Pack "`n_blocks` ResBlocks + an optional AttnBlock at the end" into one level.
Each of `down1/down2/down3/mid/up1/up2/up3` in the UNet **is a Stage**.
The first ResBlock changes the channel count; the rest keep it unchanged.

In [ ]:
class Stage(nn.Module):
    def __init__(self, in_ch, out_ch, c_dim, n_blocks=2, attn=False):
        super().__init__()
        self.blocks = nn.ModuleList([ResBlock(in_ch if i == 0 else out_ch, out_ch, c_dim)
                                     for i in range(n_blocks)])
        self.attn = AttnBlock(out_ch) if attn else None
    def forward(self, x, c):
        for b in self.blocks: x = b(x, c)       # pass through each ResBlock in turn (all receive condition c)
        if self.attn is not None: x = self.attn(x)
        return x

## 5. Text-conditioned UNet (core of this version: how semantics enter)

**Skeleton**: U-shaped. The encoder downsamples layer by layer `96→48→24→12`, channels `128→256→512`; bottleneck;
The decoder upsamples layer by layer back to `96`; **skip connections** concatenate encoder features back into the decoder's same-resolution layers (preserving detail).

**Semantic conditioning (key change)** —— no longer use `nn.Embedding` to look up by name, but instead:
- `clip_table`: precomputed **CLIP text vector table** (one row per Pokémon + a final null row), registered as a **frozen buffer** (not trained)
- `text_proj`: a small MLP that **projects** the 512-dim CLIP vector to the condition dimension `c_dim` (this is trained)
- `cond()`: add `time embedding + text_proj(CLIP vector)` = condition `c`, injected into every ResBlock
  - pass `y` (class index) -> looks up `clip_table`
  - pass `y_emb` (directly give the CLIP vector) -> used for **free text / fusion** (takes priority)
- `num_classes` = number of classes; the last row (index=num_classes) is **null**, used for classifier-free guidance

Key point: CLIP is **frozen** and only used as a lookup; the only things actually trained are `text_proj` + the UNet itself.

In [ ]:
class UNet(nn.Module):
    def __init__(self, in_ch=3, base=128, c_dim=256, n_blocks=2, clip_dim=512, clip_table=None):
        super().__init__()
        self.c_dim = c_dim
        # CLIP text embedding table (frozen buffer): (num_classes+1, clip_dim), last row=null (empty-string encoding), used for CFG
        self.num_classes = (clip_table.shape[0] - 1) if clip_table is not None else None
        if clip_table is not None:
            self.register_buffer("clip_table", clip_table)
        self.text_proj = nn.Sequential(nn.Linear(clip_dim, c_dim), nn.SiLU(), nn.Linear(c_dim, c_dim))
        self.time_mlp  = nn.Sequential(nn.Linear(c_dim, c_dim), nn.SiLU(), nn.Linear(c_dim, c_dim))

        self.in_conv = nn.Conv2d(in_ch, base, 3, padding=1)
        self.down1 = Stage(base,      base,     c_dim, n_blocks)              # 96
        self.down2 = Stage(base,      base*2,   c_dim, n_blocks)              # 48
        self.down3 = Stage(base*2,    base*4,   c_dim, n_blocks, attn=True)   # 24 (+attn)
        self.downsample = nn.AvgPool2d(2)
        self.mid = Stage(base*4, base*4, c_dim, n_blocks, attn=True)          # 12 (+attn)
        self.upsample = nn.Upsample(scale_factor=2, mode="nearest")
        self.up3 = Stage(base*4 + base*4, base*2, c_dim, n_blocks, attn=True) # 24 (+attn)
        self.up2 = Stage(base*2 + base*2, base,   c_dim, n_blocks)            # 48
        self.up1 = Stage(base   + base,   base,   c_dim, n_blocks)            # 96
        self.out = nn.Sequential(nn.GroupNorm(8, base), nn.SiLU(), nn.Conv2d(base, in_ch, 3, padding=1))

    def cond(self, t, y=None, y_emb=None):
        """Composite condition c = time embedding + text semantics.
        y: class index (looks up clip_table); y_emb: directly given CLIP vector (free text / fusion, takes priority)."""
        c = self.time_mlp(sinusoidal_embedding(t, self.c_dim))
        if y_emb is not None:
            c = c + self.text_proj(y_emb)
        elif self.num_classes is not None:
            if y is None:                                   # not given -> use null for all (unconditional)
                y = torch.full((t.size(0),), self.num_classes, device=t.device, dtype=torch.long)
            c = c + self.text_proj(self.clip_table[y])
        return c

    def forward(self, x, t, y=None, y_emb=None):
        c = self.cond(t, y, y_emb)
        x = self.in_conv(x)
        s1 = self.down1(x, c); x = self.downsample(s1)      # save skip s1, down to 48
        s2 = self.down2(x, c); x = self.downsample(s2)      # s2, down to 24
        s3 = self.down3(x, c); x = self.downsample(s3)      # s3, down to 12
        x = self.mid(x, c)                                  # bottleneck
        x = self.upsample(x); x = self.up3(torch.cat([x, s3], 1), c)   # concat s3
        x = self.upsample(x); x = self.up2(torch.cat([x, s2], 1), c)   # concat s2
        x = self.upsample(x); x = self.up1(torch.cat([x, s1], 1), c)   # concat s1
        return self.out(x)                                  # predicted noise ε

**Shape self-test** (randomly build a 21×512 CLIP table).

In [ ]:
_ct = torch.randn(21, 512); _ct = _ct / _ct.norm(dim=-1, keepdim=True)
_net = UNet(clip_table=_ct).to(device)
_x = torch.randn(2,3,96,96,device=device); _t = torch.randint(0,1000,(2,),device=device); _y = torch.randint(0,20,(2,),device=device)
print("output:", tuple(_net(_x,_t,_y).shape),
      "| trainable parameters: %.2fM" % (sum(p.numel() for p in _net.parameters() if p.requires_grad)/1e6))

## 6. Diffusion process + loss

- **Forward noising** `q_sample`: `x_t = √ᾱ_t·x0 + √(1-ᾱ_t)·ε` (in one step, no parameters)
- **loss**: let the network predict noise ε, `MSE(ε_θ(x_t,t,y), ε)` —— this is the **L_simple** from the simplified maximum-likelihood ELBO
- **CFG conditional dropout** (`p_uncond`): during training, with some probability replace `y` with null, so the network learns both "conditional/unconditional"
- **Sampling** `sample`: repeatedly denoise from pure noise for T steps; when conditioned, amplify with `guidance`:
  `ε = ε_uncond + w·(ε_cond − ε_uncond)`

In [ ]:
class Diffusion:
    def __init__(self, timesteps=1000, beta_start=1e-4, beta_end=0.02, device="cuda"):
        self.T = timesteps; self.device = device
        beta = torch.linspace(beta_start, beta_end, timesteps, device=device)
        self.beta = beta; self.alpha = 1.0 - beta
        self.alpha_bar = torch.cumprod(self.alpha, dim=0)               # ᾱ_t

    def q_sample(self, x0, t, noise):                                   # forward noising
        ab = self.alpha_bar[t][:, None, None, None]
        return ab.sqrt() * x0 + (1.0 - ab).sqrt() * noise

    def loss(self, model, x0, y, p_uncond=0.1):                        # training objective (includes CFG dropout)
        B = x0.size(0)
        t = torch.randint(0, self.T, (B,), device=x0.device)
        noise = torch.randn_like(x0)
        x_t = self.q_sample(x0, t, noise)
        if model.num_classes is not None and p_uncond > 0:
            y = y.clone(); y[torch.rand(B, device=x0.device) < p_uncond] = model.num_classes  # -> null
        return F.mse_loss(model(x_t, t, y), noise)

    @torch.no_grad()
    def sample(self, model, n, y=None, y_emb=None, guidance=3.0, img_size=96, channels=3):
        model.eval()
        x = torch.randn(n, channels, img_size, img_size, device=self.device)
        use_cfg = (model.num_classes is not None) and (y is not None or y_emb is not None)
        if use_cfg:
            null = torch.full((n,), model.num_classes, device=self.device, dtype=torch.long)
        for i in reversed(range(self.T)):
            t = torch.full((n,), i, device=self.device, dtype=torch.long)
            if use_cfg:
                eps_c = model(x, t, y=y, y_emb=y_emb); eps_u = model(x, t, y=null)
                pred = eps_u + guidance * (eps_c - eps_u)              # CFG amplifies the condition
            else:
                pred = model(x, t)
            bt, at, abt = self.beta[i], self.alpha[i], self.alpha_bar[i]
            mean = (1/at.sqrt()) * (x - (bt/(1-abt).sqrt()) * pred)
            x = mean + (bt.sqrt()*torch.randn_like(x) if i > 0 else 0.0)
        model.train()
        return x

## 7. Experiment / stage configuration

Change `STAGE` to switch stages:
- **Stage 1**: all generations `pokemon_all`, from scratch, 300 epochs, learns general semantics.
- **Stage 2**: specialize on those 20 (`pokemon20_aug`) + **rehearsal** (randomly draw `REHEARSAL_N` images from the full set and mix them in),
  Load Stage 1 weights, fine-tune with low lr.

**Two key designs:**
- **Rehearsal**: stage 2 feeds not only the 20, but also mixes in some full-set data, to **prevent catastrophic forgetting**——
  Let it sharpen the 20 while not forgetting the open semantics learned in stage one.
- **Keep both weights**: Stage 1 / Stage 2 are written to **different experiment folders**, Stage 2 will not overwrite Stage 1.
  Use Stage 1 for open text / general fusion; use Stage 2 for the clearest version of those 20.

> The class table (`clip_table`) uses the **full set of 988** in both stages, with consistent indices, so transfer is clean.

In [ ]:
STAGE = 1   # ← 1 or 2

EXP_BASE  = "/root/autodl-tmp/experiments"
ALL_DIR   = "/root/autodl-tmp/pokemon_all"        # full set (all 988 classes), also the source of rehearsal data
BASE = 128; N_BLOCKS = 2; EMA_DECAY = 0.999
IMG_SIZE = 96; BATCH_SIZE = 64; TIMESTEPS = 1000

if STAGE == 1:
    FOCUS_DIR   = ALL_DIR                          # stage one: train on the full set directly
    DATASET_TAG = "pokemonALL"
    INIT_CKPT   = None
    REHEARSAL_N = 0                                # stage one needs no rehearsal
    EPOCHS = 300; LR = 2e-4
else:
    FOCUS_DIR   = "/root/autodl-tmp/pokemon20_aug" # stage two: specialize on these 20
    DATASET_TAG = "pokemon20aug"
    INIT_CKPT   = os.path.join(EXP_BASE, "exp06_stage1_pokemonALL_clipText/checkpoints/ckpt_ep300.pt")
    REHEARSAL_N = 3000                             # rehearsal: randomly sample this many from the full set to mix in, prevents forgetting
    EPOCHS = 120; LR = 1e-4                        # smaller lr for fine-tuning

EXP_NAME = f"exp06_stage{STAGE}_{DATASET_TAG}_clipText"   # different stages write to different folders -> keep both weights
OUT_DIR  = os.path.join(EXP_BASE, EXP_NAME)
CKPT_DIR = os.path.join(OUT_DIR, "checkpoints"); SAMPLE_DIR = os.path.join(OUT_DIR, "samples"); TB_DIR = os.path.join(OUT_DIR, "tb")
for d in (CKPT_DIR, SAMPLE_DIR, TB_DIR): os.makedirs(d, exist_ok=True)
try:                                                  # symlink to gpuhub's built-in TensorBoard
    os.makedirs("/root/tf-logs", exist_ok=True)
    link = os.path.join("/root/tf-logs", EXP_NAME)
    if not os.path.exists(link): os.symlink(TB_DIR, link)
except OSError: pass
import shutil   # auto-archive: copy this script (real filename) into the current experiment directory
SCRIPT_NAME = "pokemon_train_clip.ipynb"
SCRIPT_HOME = os.path.join(EXP_BASE, "exp06_stage1_pokemonALL_clipText", SCRIPT_NAME)   # the script's home
try:
    dst = os.path.join(OUT_DIR, SCRIPT_NAME)
    if os.path.abspath(SCRIPT_HOME) != os.path.abspath(dst): shutil.copy(SCRIPT_HOME, dst)
except Exception: pass
print("experiment:", EXP_NAME, "| STAGE", STAGE, "| rehearsal count:", REHEARSAL_N)

## 8. Load precomputed CLIP vectors (training never touches CLIP)

CLIP has been run **offline** and saved as `clip_emb.pt` (988 Pokémon + null). Here it is loaded directly and assembled by the **full set of 988 classes** into
`clip_table` (last row=null, shared by both stages, consistent indices). **The training notebook does not need open_clip and does not occupy CLIP GPU memory.**

In [ ]:
blob = torch.load("/root/autodl-tmp/clip_emb.pt", map_location="cpu")
name2vec = {n: blob["emb"][i] for i, n in enumerate(blob["names"])}
null_vec = blob["null"]

# Class table = full set (all 988 classes), both stages use the same table with consistent indices
# (other Pokémon mixed in during stage-two rehearsal must also be in the table, so build it from the full set ALL_DIR here, not from FOCUS_DIR)
names = sorted({os.path.basename(p).split("__")[0] for p in glob.glob(os.path.join(ALL_DIR, "*.png"))} & set(name2vec))
clip_table = torch.stack([name2vec[n] for n in names] + [null_vec])   # (988+1, 512), last row=null
CLIP_DIM = clip_table.shape[1]; NUM_CLASSES = len(names)

json.dump(names, open(os.path.join(OUT_DIR, "classes.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=1)
torch.save(clip_table, os.path.join(OUT_DIR, "clip_table.pt"))
print("class table (full set):", NUM_CLASSES, "classes | clip_table:", tuple(clip_table.shape))

## 9. Data loading (specialization + rehearsal)

- **Specialization data** = `FOCUS_DIR` (stage 1 = full set; stage 2 = those 20)
- **Rehearsal data** = randomly draw `REHEARSAL_N` images from the full set `ALL_DIR` (stage 2 only), mixed in to prevent forgetting
- Each image returns `(image tensor, class index)`, with index aligned to the full set `names`/`clip_table`

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import random

class ImageFolderFlat(Dataset):
    def __init__(self, paths, names, img_size=96):
        self.cls2idx = {n: i for i, n in enumerate(names)}
        self.paths = [p for p in paths if os.path.basename(p).split("__")[0] in self.cls2idx]
        self.tf = transforms.Compose([transforms.Resize((img_size, img_size)),
                                      transforms.ToTensor(), transforms.Normalize([0.5]*3, [0.5]*3)])
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        p = self.paths[i]; img = Image.open(p)
        if img.mode in ("RGBA","LA","P"):
            img = Image.alpha_composite(Image.new("RGBA", img.size, (255,)*4), img.convert("RGBA")).convert("RGB")
        else: img = img.convert("RGB")
        return self.tf(img), self.cls2idx[os.path.basename(p).split("__")[0]]

focus_paths = glob.glob(os.path.join(FOCUS_DIR, "*.png"))      # specialization data (stage 1=full set; stage 2=the 20)
rehearsal_paths = []
if REHEARSAL_N > 0:                                            # rehearsal: randomly sample a batch from the full set and mix in, to prevent forgetting
    pool = glob.glob(os.path.join(ALL_DIR, "*.png"))
    random.seed(0); rehearsal_paths = random.sample(pool, min(REHEARSAL_N, len(pool)))

ds = ImageFolderFlat(focus_paths + rehearsal_paths, names, IMG_SIZE)
dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)
print(f"focus {len(focus_paths)} images + rehearsal {len(rehearsal_paths)} images = {len(ds)} images | classes {NUM_CLASSES}")

## 10. Training (EMA + CFG + TensorBoard)

- **EMA** (weight moving average, with warmup): sampling/saving weights uses smoothed weights, making sample images more stable.
  warmup `d=min(decay,(1+step)/(10+step))` prevents EMA from being stuck at random initialization for a long time on small data.
- **Stage 2**: when `INIT_CKPT` is non-empty, load Stage 1 weights (`text_proj`+UNet fully transferred;
  `clip_table` is a fixed constant table, just skip it). Combined with the **rehearsal data** above, it both sharpens the 20 and prevents forgetting.
- Every `SAMPLE_EVERY` epochs **use one free-text description** (`eval_emb.pt`, "a flying electric-type fluffy mouse")
  Sample one image + save EMA weights; write loss/sample images to TensorBoard.

> ⚠️ This cell actually starts training. First confirm everything above runs.

In [ ]:
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
import matplotlib.pyplot as plt

class EMA:
    """Exponential moving average of weights (with warmup)."""
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}
    @torch.no_grad()
    def update(self, model, step):
        d = min(self.decay, (1+step)/(10+step))            # warmup: catch up quickly early on
        for k, v in model.state_dict().items():
            s = self.shadow[k]
            s.mul_(d).add_(v.detach(), alpha=1-d) if v.dtype.is_floating_point else s.copy_(v)
    def copy_to(self, model): model.load_state_dict(self.shadow, strict=True)

writer = SummaryWriter(TB_DIR)
model     = UNet(base=BASE, n_blocks=N_BLOCKS, clip_dim=CLIP_DIM, clip_table=clip_table).to(device)
if INIT_CKPT:                                              # Stage 2: transfer Stage 1 weights (skip clip_table)
    sd = torch.load(INIT_CKPT, map_location=device)
    sd = {k: v for k, v in sd.items() if not k.startswith("clip_table")}
    miss = model.load_state_dict(sd, strict=False)
    print("Initialized from Stage1, missing keys:", list(miss.missing_keys))
ema       = EMA(model, EMA_DECAY)
ema_model = UNet(base=BASE, n_blocks=N_BLOCKS, clip_dim=CLIP_DIM, clip_table=clip_table).to(device)  # for sampling only
diff = Diffusion(timesteps=TIMESTEPS, device=device)
opt = torch.optim.AdamW(model.parameters(), lr=LR)
scaler = torch.amp.GradScaler("cuda")
print("trainable params: %.2fM" % (sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6))

# Validation images are generated from one [free-text description] (CLIP vectors are encoded and stored offline, training never touches CLIP)
EVAL_PROMPT = "a fluffy electric-type mouse-shaped pokémon with wings"
eval_emb = torch.load("/root/autodl-tmp/eval_emb.pt", map_location=device).view(1, -1)  # (1, 512)
SAMPLE_EVERY = 10

step = 0
for epoch in range(1, EPOCHS+1):
    model.train()
    pbar = tqdm(dl, desc=f"[S{STAGE}] epoch {epoch}/{EPOCHS}")
    for x0, y in pbar:
        x0 = x0.to(device, non_blocking=True); y = y.to(device, non_blocking=True)
        opt.zero_grad()
        with torch.amp.autocast("cuda"):
            loss = diff.loss(model, x0, y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        ema.update(model, step); writer.add_scalar("train/loss", loss.item(), step); step += 1
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    if epoch % SAMPLE_EVERY == 0 or epoch == EPOCHS:
        ema.copy_to(ema_model)
        img = (diff.sample(ema_model, 1, y_emb=eval_emb, guidance=3.0, img_size=IMG_SIZE).clamp(-1,1)+1)/2
        plt.figure(figsize=(2.6,2.8)); plt.imshow(img[0].cpu().permute(1,2,0).numpy())
        plt.axis("off"); plt.title("eval: fluffy electric winged mouse", fontsize=8)
        plt.savefig(os.path.join(SAMPLE_DIR, f"sample_ep{epoch}.png"), dpi=120, bbox_inches="tight")
        writer.add_figure("samples", plt.gcf(), epoch); plt.close()
        torch.save(ema.shadow, os.path.join(CKPT_DIR, f"ckpt_ep{epoch}.pt"))
        print(f"  ✔ ep{epoch}: sample image (description-generated) + ckpt saved -> {EXP_NAME}")
writer.close()

## ✅ Next steps

- After Stage 1 finishes → change `STAGE` to **2** and rerun (automatically loads Stage 1 weights to fine-tune those 20).
- Generation / fusion / free text → use `pokemon_gen_clip.ipynb`.
- For a quick check: temporarily reduce `EPOCHS` (e.g. 20) to first see whether the loss drops.